# Imports

In [1]:
import boto3
import re
import os
import random
from botocore.config import Config
from IPython.display import HTML, display
import pandas as pd
from tqdm import tqdm
import json
import html
from bedrock import (
    extract_skills_from_text,
    extract_career_info,
    extract_education,
    BedrockConfig
)
from utils import (
    extract_summary_section,
    remove_education_section,
    extract_candidate_contact_info
)

from dotenv import load_dotenv
load_dotenv()

True

# Configurations

In [2]:
# --- Data Configuration ---
limit = 1

# --- Core Project Configuration ---
REGION = os.environ.get("AWS_DEFAULT_REGION", "eu-west-1")

# --- Initialize AWS Clients ---
boto_config = Config(
    retries={'max_attempts': 5, 'mode': 'standard'},
    max_pool_connections=50 
)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=REGION, config=boto_config)

# --- LLM Configuration ---
MODEL_NAME = "google.gemma-3-4b-it"
MAX_GENERATION_TOKENS = 768

config = BedrockConfig(
    bedrock_runtime=bedrock_runtime,
    model_name=MODEL_NAME,
    max_generation_tokens=MAX_GENERATION_TOKENS,
)

# Load Dataset

In [3]:
file_name = "Entity Recognition in Resumes.json"
file_path = f"../data/{file_name}"
texts = []

EDU_HEADER_PATTERN = re.compile(
    r"(?i)(education|academics|academic background|qualifications)\b"
)
DATE_LINE_PATTERN = re.compile(
    r"(?i)\b(?:(?:19|20)\d{2}\b|(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+(?:19|20)\d{2})"
)

def inject_synthetic_gpa(text: str) -> str:
    """Randomly injects a synthetic GPA (50% chance) into the education block under the date."""
    if random.random() > 0.5:
        return text

    gpa_val = random.randint(70, 95)
    gpa_line = f"GPA: {gpa_val} / 100"

    lines = text.splitlines()
    edu_start_idx = None

    # 1. Locate the start of the EDUCATION section
    for idx, line in enumerate(lines):
        if EDU_HEADER_PATTERN.search(line):
            edu_start_idx = idx
            break

    # If no education section found, leave text untouched
    if edu_start_idx is None:
        return text

    # 2. Look for the first date line under the education section
    inserted = False
    for idx in range(edu_start_idx + 1, len(lines)):
        # Stop if another major section begins (heuristic: uppercase section header)
        if (
            lines[idx].isupper()
            and len(lines[idx].strip()) > 3
            and idx > edu_start_idx + 2
        ):
            break

        if DATE_LINE_PATTERN.search(lines[idx]):
            lines.insert(idx + 1, gpa_line)
            inserted = True
            break

    # If no explicit date was found, append right under the education header
    if not inserted:
        lines.insert(edu_start_idx + 1, gpa_line)

    return "\n".join(lines)


with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if limit is not None and len(texts) >= limit:
            break

        if not line.strip():
            continue

        data = json.loads(line)
        resume_text = data.get("content", "")
        if resume_text.strip():
            # Apply 50% GPA injection logic
            modified_text = inject_synthetic_gpa(resume_text)
            texts.append(modified_text)

df = pd.DataFrame({"text": texts})

print(f"Total Resumes Loaded: {len(df)}")

Total Resumes Loaded: 1


In [4]:
with open(f"../data/my_resumes.json", "r", encoding="utf-8") as f:
    new_texts = json.load(f)

my_resumes_df = pd.DataFrame({"text": new_texts})
my_resumes_df = my_resumes_df.dropna(subset=["text"])
df = pd.concat([df, my_resumes_df], ignore_index=True)

print(f"Total resumes after concatenation: {len(df)}")

Total resumes after concatenation: 5


# Extract Info From Resume

In [6]:
tqdm.pandas()

# ==========================================
# 1. TOGGLE FLAGS
# ==========================================
RUN_SKILLS = False
RUN_CAREER = True
RUN_EDUCATION = False
RUN_SUMMARY = True
RUN_CONTACT = True

# ==========================================
# 2. SELECTIVE WORKER
# Only invokes Bedrock APIs / extractors for enabled tasks
# ==========================================
def process_single_resume(
    text: str,
    run_skills: bool = True,
    run_career: bool = True,
    run_education: bool = True,
    run_summary: bool = True,
    run_contact: bool = True,
) -> dict:
    result = {}

    # 1. Contact info extraction (Full Name, Phone, Email, GitHub, LinkedIn)
    if run_contact:
        try:
            result["contact_info"] = extract_candidate_contact_info(text)
        except Exception:
            result["contact_info"] = {
                "full_name": None,
                "email": None,
                "phone": None,
                "linkedin": None,
                "github": None,
            }

    # 2. Skills extraction uses the complete text
    if run_skills:
        try:
            result["skills"] = extract_skills_from_text(text, config)
        except Exception:
            result["skills"] = []

    # 3. Career extraction receives text stripped of the Education section
    if run_career:
        try:
            # Strip education section to prevent academic role leakage
            edu_removal_res = remove_education_section(text)
            career_input_text = edu_removal_res.get("cleaned_text", text)

            career_info = extract_career_info(career_input_text, config)
            result["job_title"] = career_info.get("job_title")
            result["experience"] = career_info.get("experience", [])
        except Exception:
            result["job_title"] = None
            result["experience"] = []

    # 4. Education extraction uses the complete text
    if run_education:
        try:
            result["education"] = extract_education(text, config)
        except Exception:
            result["education"] = []

    # 5. Summary extraction uses the complete text
    if run_summary:
        try:
            result["summary"] = extract_summary_section(text)["summary_text"]
        except Exception:
            result["summary"] = ""

    return result


# ==========================================
# 3. RUN EXTRACTION & MERGE
# ==========================================
active_flags = [RUN_SKILLS, RUN_CAREER, RUN_EDUCATION, RUN_SUMMARY, RUN_CONTACT]

if not any(active_flags):
    print("No extractors selected. Set at least one toggle flag to True.")
else:
    # Run extraction with the active toggles
    extracted_series = df["text"].progress_apply(
        lambda txt: process_single_resume(
            txt,
            run_skills=RUN_SKILLS,
            run_career=RUN_CAREER,
            run_education=RUN_EDUCATION,
            run_summary=RUN_SUMMARY,
            run_contact=RUN_CONTACT,
        )
    )

    extracted_df = pd.DataFrame(extracted_series.tolist(), index=df.index)

    # Overwrite / add only the columns that were actually computed
    for col in extracted_df.columns:
        df[col] = extracted_df[col]

    print(
        f"Extraction complete! Updated columns: {list(extracted_df.columns)}"
    )

100%|██████████| 5/5 [00:04<00:00,  1.03it/s]

Extraction complete! Updated columns: ['contact_info', 'job_title', 'experience', 'summary']


# Inspect Extraction

In [10]:
# ==========================================
# 1. SELECT RESUME RECORD
# ==========================================
target_index = 0
row = df.iloc[target_index]

resume_text = row.get("text", "")
safe_text = html.escape(str(resume_text))

skills = row.get("skills") or []
title = (row.get("job_title") or "").strip()
exps = row.get("experience") or []
edus = row.get("education") or []
summary = (row.get("summary") or "").strip()
contact = row.get("contact_info") or {}

# ==========================================
# 2. CONSOLE OVERVIEW (OPTION 1)
# ==========================================
print(f"==================================================")
print(f"               RESUME #{target_index} OVERVIEW")
print(f"==================================================")
print(f"Candidate Name   : {contact.get('full_name') or 'N/A'}")
print(f"Headline Title   : {title or 'None'}")

# --- Contact Information ---
print(f"\n--- CONTACT INFORMATION ---")
print(f"  Email    : {contact.get('email') or 'N/A'}")
print(f"  Phone    : {contact.get('phone') or 'N/A'}")
print(f"  LinkedIn : {contact.get('linkedin') or 'N/A'}")
print(f"  GitHub   : {contact.get('github') or 'N/A'}")

# --- Professional Summary ---
print(f"\n--- PROFESSIONAL SUMMARY ---")
if not summary:
    print("  (No summary found)")
else:
    print(f"  {summary}")

print(f"\nSkills Extracted : {len(skills)}")
if skills:
    print(f"  Sample Skills  : {', '.join(skills[:8])}{'...' if len(skills) > 8 else ''}")

# --- Roles / Work Experience ---
print(f"\n--- WORK EXPERIENCE ({len(exps)}) ---")
if not exps:
    print("  (No roles found)")
else:
    for i, exp in enumerate(exps, 1):
        r_name = exp.get("role", "N/A")
        r_dates = exp.get("dates", "N/A")
        r_years = exp.get("years", 0.0)
        
        print(f"  {i}. {r_name}")
        print(f"     Dates: {r_dates} | Duration: {r_years} yrs")

# --- Education Entries ---
print(f"\n--- EDUCATION ENTRIES ({len(edus)}) ---")
if not edus:
    print("  (No education records found)")
else:
    for i, edu in enumerate(edus, 1):
        deg_type = edu.get("degree_type") or ""
        deg_name = edu.get("degree_name") or ""
        full_deg = f"{deg_type} - {deg_name}".strip(" -")
        inst = edu.get("institution_type") or "N/A"
        gpa = edu.get("gpa") or "N/A"
        date = edu.get("date") or "N/A"
        status = edu.get("degree_status") or "N/A"
        year = edu.get("current_study_year")
        semesters = edu.get("semesters_left")

        print(f"  {i}. {full_deg or 'Unknown Degree'}")
        print(f"     Institution: {inst}")
        print(f"     GPA: {gpa} | Dates: {date}")
        print(f"     Status: {status} (Study Year: {year}, Semesters Left: {semesters})")

print(f"==================================================\n")

# ==========================================
# 3. BUILD ENTITY TARGETS FOR HIGHLIGHTING
# ==========================================
has_overlap = any(
    x.get("role", "").strip().lower() == title.lower() for x in exps
)

roles = [
    x["role"].strip()
    for x in exps
    if x.get("role") and x["role"].strip().lower() != title.lower()
]

exp_dates = [
    x["dates"].strip()
    for x in exps
    if x.get("dates") and x["dates"].strip().lower() != "no dates provided"
]

institutions = [
    e["institution_type"].strip()
    for e in edus
    if e.get("institution_type") and e["institution_type"].strip().lower() != "other"
]

degrees = [e["degree_type"].strip() for e in edus if e.get("degree_type")]
majors = [e["degree_name"].strip() for e in edus if e.get("degree_name")]
gpas = [
    str(e["gpa"]).strip()
    for e in edus
    if e.get("gpa") and str(e["gpa"]).strip().lower() != "none"
]
edu_dates = [
    str(e["date"]).strip()
    for e in edus
    if e.get("date") and str(e["date"]).strip().lower() != "none"
]

# (Items, Tag Label, Background Hex, Text Color Hex)
targets = [
    (skills, "[Skill]", "#ffeb3b", "#000000"),
    ([title] if title else [], "[Title & Role]" if has_overlap else "[Title]", "#0288d1", "#ffffff"),
    (roles, "[Role]", "#2e7d32", "#ffffff"),
    (exp_dates, "[Exp Date]", "#8e24aa", "#ffffff"),
    (institutions, "[Institution]", "#00838f", "#ffffff"),
    (degrees, "[Degree]", "#f57c00", "#ffffff"),
    (majors, "[Major]", "#00897b", "#ffffff"),
    (gpas, "[GPA]", "#c2185b", "#ffffff"),
    (edu_dates, "[Edu Date]", "#5c6bc0", "#ffffff"),
]

# ==========================================
# 4. APPLY HTML HIGHLIGHTS & BUILD LEGEND
# ==========================================
legend_items = []

for items, label, bg_color, text_color in targets:
    clean_items = [s for s in items if s and str(s).strip()]
    
    # Sort phrases by length descending to match longest sequences first
    unique_items = sorted(set(clean_items), key=len, reverse=True)
    
    if unique_items:
        escaped_items = [re.escape(html.escape(s)) for s in unique_items]
        
        # Word boundary lookahead/behind avoiding intra-tag HTML match
        pattern = rf"(?i)(?<![\w#+])({'|'.join(escaped_items)})(?![\w#+])(?![^<]*>)"
        
        safe_text = re.sub(
            pattern,
            rf'<mark style="background:{bg_color};color:{text_color};padding:2px 5px;border-radius:4px;font-weight:bold">\1 {label}</mark>',
            safe_text,
        )
        
        legend_items.append(
            f'<span style="background:{bg_color};color:{text_color};padding:4px 8px;border-radius:4px;font-weight:bold;margin-right:8px;margin-bottom:6px;display:inline-block;border:1px solid rgba(255,255,255,0.15)">■ {label.strip("[]")}</span>'
        )

# ==========================================
# 5. RENDER DARK-THEME NOTEBOOK VIEWER
# ==========================================
display(
    HTML(
        f"""
    <div style="margin-bottom:14px;font-family:sans-serif;font-size:12px;background:#252526;padding:12px;border-radius:6px;border:1px solid #444;display:flex;flex-wrap:wrap;">
        <div style="width:100%;margin-bottom:8px;color:#eee;font-weight:bold;font-size:13px;">Extracted Entity Highlights:</div>
        {''.join(legend_items)}
    </div>
    <div style="white-space:pre-wrap;font-family:monospace;font-size:13px;line-height:1.7;background:#1e1e1e;color:#d4d4d4;padding:20px;border-radius:6px;border:1px solid #333;box-shadow:0 4px 6px rgba(0,0,0,0.3);">{safe_text}</div>
"""
    )
)

               RESUME #0 OVERVIEW
Candidate Name   : Abhishek Jha
Headline Title   : Application Development Associate

--- CONTACT INFORMATION ---
  Email    : N/A
  Phone    : N/A
  LinkedIn : N/A
  GitHub   : N/A

--- PROFESSIONAL SUMMARY ---
  (No summary found)

Skills Extracted : 0

--- WORK EXPERIENCE (1) ---
  1. Application Development Associate
     Dates: November 2017 to Present | Duration: 9.0 yrs

--- EDUCATION ENTRIES (0) ---
  (No education records found)



# Download Dataframe

In [16]:
df.dropna().to_csv("resumes_test_set.csv", index=False)